In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio as rio
from rasterio.warp import reproject, Resampling
from matplotlib import pyplot as plt
from pathlib import Path

In [9]:
ROOT = Path.cwd().parent
DATA = ROOT / "data"
RESULTS = ROOT / "results"
FEATURE_DIR = RESULTS / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
GLiM_ = rio.open(DATA / 'GLiM.tif')

In [4]:
country_raster_1km_ = rio.open(RESULTS / 'country-rasters' / 'country-raster-eck4-1km.tif')

In [5]:
GLiM = GLiM_.read(1)

In [6]:
GLiM_1km = np.zeros(country_raster_1km_.shape, dtype=np.uint8)

In [7]:
reproject(
    source=GLiM,
    destination=GLiM_1km,
    src_transform=GLiM_.transform,
    src_crs=GLiM_.crs,
    dst_transform=country_raster_1km_.transform,
    dst_crs=country_raster_1km_.crs,
    resampling=Resampling.nearest
)

(array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(16922, 33844), dtype=uint8),
 Affine(1000.0, 0.0, -16922000.0,
        0.0, -1000.0, 8461000.0))

In [8]:
class_names = dict({
    1: 'su',
    2: 'vb',
    3: 'ss', 
    4: 'pb', 
    5: 'sm', 
    6: 'sc', 
    7: 'va', 
    8: 'mt', 
    9: 'pa', 
    10: 'vi', 
    11: 'wb', 
    12: 'py', 
    13: 'pi', 
    14: 'ev', 
    15: 'nd', 
    16: 'ig'
})

In [11]:
for i in range(1, 17):
    a_class = class_names[i]
    print(a_class)
    one_hot_mask_ = GLiM_1km == i
    one_hot_mask = one_hot_mask_.astype(np.uint8)
    output_path = FEATURE_DIR / f'{a_class}-1km.tif'
    with rio.open(
        output_path,
        "w",
        driver="GTiff",
        height=one_hot_mask.shape[0],
        width=one_hot_mask.shape[1],
        count=1,
        dtype=one_hot_mask.dtype,
        crs=country_raster_1km_.crs,
        transform=country_raster_1km_.transform,
        compress='lzw',
    ) as dst:
        dst.write(np.array([one_hot_mask]))

su
vb
ss
pb
sm
sc
va
mt
pa
vi
wb
py
pi
ev
nd
ig
